# Research on AAC Vocabulary and Auto Generation



##0. Background Knowledge


Understanding what is semantic search:

Quick understanding of  Word embedding:

https://www.youtube.com/watch?v=lPTcTh5sRug

https://www.youtube.com/shorts/FJtFZwbvkI4

Tutorial

https://oneuptime.com/blog/post/2025-08-21-vector-embeddings/view


## 1. Load Vocabulary

In [ ]:
# Load the AAC research vocabulary
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

csv_path = '/content/drive/MyDrive/corpus_vocabulary_clean.csv'


import pandas as pd
df = pd.read_csv(csv_path)

name_to_idx = {n.lower(): i for i, n in enumerate(df['name_en'])}

for c in sorted(df['category'].unique()):
  print(c)

print(f'categories: {df["category"].nunique()}   total words: {len(df)}')
df.head()

Mounted at /content/drive
animal
color
core
disability
food
health
hobby
home
human
life
marriage
nation
others
people
religion
school
service
sports
time
traffic
university
work
categories: 22   total words: 1998


,category,name_en,description_brief
0,animal,Alligator,A cartoonish alligator primarily green with a ...
1,animal,Anaconda,The illustration depicts a brown snake or serp...
2,animal,Anchovies,The illustration features four stylized fish l...
3,animal,Ant,"A simplified, stylized illustration of an ant ..."
4,animal,Anteater,"A stylized, cartoon-like drawing of an anteate..."


In [ ]:
# Step 1+2: Fetch all ARASAAC pictograms and extract single and compound words
# Free — no API key needed, no tokens used
import requests, json, time

CORPUS_PATH = '/content/drive/MyDrive/corpus_vocabulary_clean.csv'

SLOTS = [
    'core', 'action', 'feeling', 'repair', 'need', 'people',
    'topic_school', 'topic_meals', 'topic_play', 'topic_clinic',
    'topic_transitions', 'general'
]

print("Fetching ARASAAC pictograms (free public API, ~30s)...")
resp = requests.get("https://api.arasaac.org/api/pictograms/all/en", timeout=120)
pictograms = resp.json()
print(f"  Received {len(pictograms)} pictograms")

seen = set()
arasaac_words = []
single_count = 0
compound_count = 0
dropped_count = 0

for picto in pictograms:
    for kw in picto.get("keywords", []):
        word = kw.get("keyword", "").strip()
        parts = word.split()
        if not word:
            continue
        if len(parts) <= 2:   # keep single words AND 2-word compounds (hot dog, ice cream)
            lower = word.lower()
            if lower not in seen:
                seen.add(lower)
                arasaac_words.append(word)
                if len(parts) == 1:
                    single_count += 1
                else:
                    compound_count += 1
        else:
            dropped_count += 1  # 3+ word phrases dropped (e.g. "basic computer skills")

print(f"  Single words  : {single_count}")
print(f"  2-word compounds: {compound_count}  (e.g. hot dog, ice cream, school bus)")
print(f"  Dropped (3+ words): {dropped_count}")
print(f"  Total kept    : {len(arasaac_words)}")
print("  (No tokens used — run Step 3 next)")

In [ ]:
# Step 3: Dedupe ARASAAC words against existing corpus
existing_lower = set(df['name_en'].str.lower())

new_rows = [
    {'category': '', 'name_en': w, 'description_brief': ''}
    for w in arasaac_words
    if w.lower() not in existing_lower
]

merged = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
print(f"  Existing words : {len(df)}")
print(f"  New from ARASAAC: {len(new_rows)}")
print(f"  Total merged   : {len(merged)}")
print(f"\nEstimated API calls for full reclassification: ~{len(merged)//50}")

In [ ]:
# Step 4a: Dry-run — classifies the first 50 words only (1 API call)
# Inspect the output before committing to the full loop in Step 4b

SYSTEM_PROMPT = """Classify each word into exactly one AAC vocabulary slot.
Return ONLY a JSON object mapping word → slot. No explanation.

Slots:
- core: high-frequency words used constantly (want, need, go, stop, help, more, finished, yes, no, I, you)
- action: verbs (eat, drink, play, read, swim, build, make, draw, run, jump)
- feeling: emotional/state words (happy, sad, tired, scared, frustrated, excited, calm)
- repair: conversation repair (different, again, wait, stop, oops, no, wrong)
- need: self-advocacy (bathroom, break, quiet, food, drink, rest, help)
- people: people/roles (mom, dad, friend, teacher, doctor, baby, sister)
- topic_school: school context (homework, pencil, recess, math, class, book, backpack)
- topic_meals: meal context (pizza, juice, snack, fork, plate, hungry, lunch, cookie)
- topic_play: play context (game, toy, ball, win, fun, turn, lego, puzzle)
- topic_clinic: medical context (hurt, pain, medicine, bandage, sick, hospital, shot)
- topic_transitions: movement/time words (home, car, next, first, then, later, now, wait, bus)
- general: everything else"""

def classify_batch(words):
    try:
        resp = arasaac_client.chat.completions.create(
            model='google/gemini-2.5-flash-lite',
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': json.dumps(words)},
            ],
            response_format={'type': 'json_object'},
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        print(f"  batch error: {e}")
        return {}

dry_run_words = merged['name_en'].tolist()[:50]
dry_run_result = classify_batch(dry_run_words)

print("Dry-run classifications (first 50 words):")
for word, slot in dry_run_result.items():
    print(f"  {word:25s} → {slot}")

print(f"\n{len(dry_run_result)}/50 words classified.")
print("If the slots look wrong, edit SYSTEM_PROMPT above and re-run this cell.")
print("If they look right, run Step 4b.")

In [ ]:
# Step 4b: Full reclassification loop — only run after dry-run looks good
# Checkpoints every 1000 words in case Colab disconnects
all_words = merged['name_en'].tolist()
categories = {}
BATCH_SIZE = 50

print(f"Classifying {len(all_words)} words (~{len(all_words)//BATCH_SIZE} API calls)...")
for i in range(0, len(all_words), BATCH_SIZE):
    batch = all_words[i:i + BATCH_SIZE]
    result = classify_batch(batch)
    categories.update(result)

    if i % 1000 == 0 and i > 0:
        # Checkpoint save — recover from here if Colab disconnects
        checkpoint = merged.copy()
        checkpoint['category'] = checkpoint['name_en'].map(lambda w: categories.get(w, ''))
        checkpoint.to_csv(CORPUS_PATH + '.checkpoint', index=False)
        print(f"  {i}/{len(all_words)} — checkpoint saved")

    time.sleep(0.3)

merged['category'] = merged['name_en'].map(lambda w: categories.get(w, 'general'))
missed = sum(1 for w in all_words if w not in categories)
print(f"\nDone. {missed} words fell back to 'general' (not in any batch response)")

In [ ]:
# Step 5: Validation gate — check slot distribution before saving
total = len(merged)
dist = merged['category'].value_counts()
general_pct = dist.get('general', 0) / total * 100

print(f"Total rows: {total}\n")
print("Slot distribution:")
print(dist.to_string())

print(f"\ngeneral%: {general_pct:.1f}%")

if general_pct > 60:
    print("\n⚠ WARNING: >60% landed in 'general' — prompt likely needs tuning.")
    print("  Do NOT run Section 2 yet. Fix the classification prompt in Cell 5b and re-run.")
else:
    print("\n✓ Distribution looks reasonable — run the Save cell next.")
    print("\nSpot-checks (5 words per slot):")
    for slot in SLOTS:
        subset = merged[merged['category'] == slot]['name_en'].tolist()
        print(f"  {slot:22s}: {subset[:5]}")

In [ ]:
# Step 6: Save merged CSV — only runs after validation passes in Step 5
merged.to_csv(CORPUS_PATH, index=False)
print(f"Saved {len(merged)} rows → {CORPUS_PATH}")
print("Run Section 2 now to re-generate embeddings.")

# Keep df and name_to_idx in sync for the rest of this session
df = merged
name_to_idx = {n.lower(): i for i, n in enumerate(df['name_en'])}

##2. Embed Vocabulary

In [ ]:
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path

from sentence_transformers import SentenceTransformer


embed_path = '/content/drive/MyDrive/corpus_vocabulary_clean.npz'

names = df["name_en"].astype(str).tolist()
print(f"  rows:        {len(df)}")

model = SentenceTransformer("all-MiniLM-L6-v2")
dim = model.get_sentence_embedding_dimension()
print(f"  embedding dim: {dim}")

t0 = time.time()
emb = model.encode(
        names,
        batch_size=128,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
)
elapsed = time.time() - t0
emb = emb.astype(np.float32)

print(f"encoded {len(names)} names in {elapsed:.1f}s")
print(f"  shape:   {emb.shape}")
print(f"  dtype:   {emb.dtype}")

norms = np.linalg.norm(emb, axis=1)
print(f"  L2 norm: min={norms.min():.4f} max={norms.max():.4f} mean={norms.mean():.4f}")

np.savez(
        embed_path,
        emb=emb,
        names=np.array(names),
        categories=df["category"].astype(str).to_numpy(),
)

size_mb = Path(embed_path).stat().st_size / 1024 / 1024
print(f"saved →  {embed_path}  ({size_mb:.1f} MB)")

# Save .bin and .meta.json for corpus.ts
bin_path  = '/content/drive/MyDrive/minilm_name_embeddings.bin'
meta_path = '/content/drive/MyDrive/minilm_name_embeddings.meta.json'
emb.tofile(bin_path)
with open(meta_path, 'w') as f:
    json.dump({"rows": len(df), "dim": emb.shape[1], "dtype": "float32"}, f)
print(f"saved →  {bin_path}")
print(f"saved →  {meta_path}")

  rows:        1998


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  embedding dim: 384


/tmp/ipykernel_854/1288465333.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

encoded 1998 names in 7.9s
  shape:   (1998, 384)
  dtype:   float32
  L2 norm: min=1.0000 max=1.0000 mean=1.0000
saved →  /content/drive/MyDrive/corpus_vocabulary_clean.npz  (3.2 MB)
saved →  /content/drive/MyDrive/minilm_name_embeddings.bin
saved →  /content/drive/MyDrive/minilm_name_embeddings.meta.json


## 3. Load Embeded

In [ ]:
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer('all-MiniLM-L6-v2')

def encode(text: str) -> np.ndarray:
    return encoder.encode([text], normalize_embeddings=True, convert_to_numpy=True)[0]


ROOT = '/content/drive/MyDrive'
VOCAB_CSV = f'{ROOT}/corpus_vocabulary_clean.csv'
EMB_NPZ   = embed_path



df = pd.read_csv(VOCAB_CSV)
df['category'] = df['category'].astype('category')

arc = np.load(EMB_NPZ, allow_pickle=True)
emb       = arc['emb']                              # L2-normalized
emb_names = arc['names']                            # parallel to df row order
emb_cats  = arc['categories']

assert len(df) == emb.shape[0], 'CSV rows and embedding rows must match'
print(f'corpus rows:    {len(df)}')
print(f'categories:     {df["category"].nunique()}')
print(f'embedding dim:  {emb.shape[1]}   ({emb.dtype})')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


corpus rows:    1998
categories:     22
embedding dim:  384   (float32)


## 4. Generate LLM response

In [ ]:
import re, yaml
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get('OPENROUTER_API_KEY'),
    base_url='https://openrouter.ai/api/v1',
)


# Fixed emotion whitelist — matches data/default_emotion_cards.yml
EMOTIONS = ['joyful', 'glad', 'happy', 'excited', 'sad', 'angry',
            'surprised', 'bored', 'tired', 'afraid', 'worried', 'tough']

_FENCE_RE = re.compile(r'```(?:[a-zA-Z0-9_+-]*)\s*\n?(.*?)\n?\s*```', re.DOTALL)


def keywords_for(
    parent_sentence: str,
    parent_type: str = 'mother',
    topic_desc: str = "The dyad gets to know what the child did on that day.",
) -> dict:
    """Send the parent sentence to the model via OpenRouter and return parsed YAML."""
    system_prompt = (
        f"You suggest English keywords for an AAC card UI. "
        f"Child age 5–7 with ASD, talking with their {parent_type}. "
        f"Conversation: {topic_desc}\n"
        f"Given the parent's last message, output 4 topic nouns, 4 action verbs, "
        f"and 4 emotions chosen from this fixed list: {', '.join(EMOTIONS)}.\n"
        f"Output ONLY this YAML, nothing else:\n"
        f"topics: [w1, w2, w3, w4]\n"
        f"actions: [w1, w2, w3, w4]\n"
        f"emotions: [w1, w2, w3, w4]"
    )
    resp = client.chat.completions.create(
        model='google/gemini-2.5-flash-lite',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': parent_sentence},
        ],
    )
    text = resp.choices[0].message.content.strip()
    print("response from model:", text)
    # Strip markdown code fences if the model wrapped the response
    m = _FENCE_RE.search(text)
    if m: text = m.group(1).strip()
    return yaml.safe_load(text)

In [ ]:
# Try a sentence
sentence = 'How was your day at school?'
result = keywords_for(sentence)
print(result)

print(f'parent: {sentence!r}\n')
for cat, words in result.items():
    print(f'  {cat:8s}: {words}')

response from model: topics: [school, friends, teacher, recess]
actions: [play, learn, talk, eat]
emotions: [happy, excited, tired, bored]
{'topics': ['school', 'friends', 'teacher', 'recess'], 'actions': ['play', 'learn', 'talk', 'eat'], 'emotions': ['happy', 'excited', 'tired', 'bored']}
parent: 'How was your day at school?'

  topics  : ['school', 'friends', 'teacher', 'recess']
  actions : ['play', 'learn', 'talk', 'eat']
  emotions: ['happy', 'excited', 'tired', 'bored']


## 5. Semantic Search for the words from vacabulary list

In [ ]:
def search_smart(query, k=10):
  s = query.lower().strip()

  # 1. exact full-string hit? — return it without embedding
  i = name_to_idx.get(s)
  if i is not None:
      return df.iloc[[i]][['category','name_en']].assign(cos=1.0, mode='exact')

  # 2. cosine + boost handles substring/word hits if any exist, otherwise pure cosine
  sims = emb @ encode(query)
  pat = rf'\b{re.escape(s)}\b'
  mask = df['name_en'].str.lower().str.contains(pat, regex=True, na=False).values
  score = sims + 0.15 * mask
  idx = np.argsort(-score)[:k]
  return df.iloc[idx][['category','name_en']].assign(
          cos=sims[idx].round(3),
          mode=np.where(mask[idx], 'word', 'cos'),
      )


print('\n=== matching corpus words (MiniLM top-1) ===')
for cat, words in result.items():
      print(f'\n{cat.upper()}')
      for w in words:
          top1 = search_smart(w, k=1).iloc[0]                # take the first row
          marker = {'exact': '★', 'word': '·', 'cos': ' '}[top1['mode']]
          print(f"  {w!r:18s} →{marker}cos={float(top1['cos']):+.3f}  "
          f"[{top1['category']:>10s}]  {top1['name_en']}  ({top1['mode']})")


=== matching corpus words (MiniLM top-1) ===

TOPICS
  'school'           →★cos=+1.000  [      core]  School  (exact)
  'friends'          →★cos=+1.000  [    sports]  Friends  (exact)
  'teacher'          →★cos=+1.000  [   traffic]  Teacher  (exact)
  'recess'           → cos=+0.493  [university]  Dormitory  (cos)

ACTIONS
  'play'             → cos=+0.641  [      core]  Game  (cos)
  'learn'            → cos=+0.640  [    school]  Education  (cos)
  'talk'             → cos=+0.696  [     hobby]  Chatting  (cos)
  'eat'              →★cos=+1.000  [    others]  Eat  (exact)

EMOTIONS
  'happy'            → cos=+0.535  [    health]  Smile!  (cos)
  'excited'          → cos=+0.523  [    animal]  Waiting  (cos)
  'tired'            → cos=+0.824  [    school]  Sleepy  (cos)
  'bored'            → cos=+0.535  [    others]  Asleep  (cos)
